# LeetCode 255 — Verify Preorder Sequence in Binary Search Tree bằng Python

Link đề: https://leetcode.com/problems/verify-preorder-sequence-in-binary-search-tree/ · Độ khó: **Medium** (Premium) · Chủ đề: Stack, Monotonic Stack, Tree, Binary Search Tree, Recursion

Skeleton LeetCode:

```python
class Solution:
    def verifyPreorder(self, preorder: List[int]) -> bool:
        ...
```

Ghi chú: đề bài viết theo hiểu biết về bài (bài Premium), nên đối chiếu lại nếu có tài khoản Premium; ví dụ và constraints lấy từ file đề có sẵn. Mọi đáp án trong notebook đều được kiểm chứng bằng code và đối chiếu với brute force độc lập (chèn lần lượt các phần tử vào một BST rồi so thứ tự duyệt tiền thứ tự).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho một dãy số **đôi một khác nhau** `preorder`, kiểm tra nó có thể là **kết quả duyệt tiền thứ tự (preorder)** của một **cây nhị phân tìm kiếm (BST)** nào đó không.
2. Nắm **tính chất BST** (con trái `<` gốc `<` con phải, cho cả cây con) và **thứ tự preorder** (gốc, rồi cây con trái, rồi cây con phải).
3. Đi từ cách **tái dựng đệ quy bằng cách tách dãy thành "nhỏ hơn gốc" và "lớn hơn gốc"** `O(n²)` → **đệ quy với cận `(lo, hi)`** `O(n)` → **stack đơn điệu + cận dưới `low`** `O(n)` → **tái sử dụng chính mảng làm stack** (`O(1)` bộ nhớ phụ, follow-up).
4. Hiểu ý then chốt: khi đi từ một phần tử xuống một phần tử **lớn hơn**, ta **rẽ phải**, và từ đó mọi phần tử sau **phải lớn hơn cha vừa bị bỏ lại** (cận dưới `low`).
5. Tránh các lỗi kinh điển: chỉ so sánh **hai phần tử liền kề**, quên cập nhật `low` khi `pop`, dùng `<=`/`<` sai, kiểm tra **inorder** thay vì preorder, và tách dãy sai chỗ ở cách đệ quy.

## 2. Đề bài & ví dụ

Cho mảng các số nguyên **đôi một khác nhau** `preorder`, trả về `true` nếu đó là **dãy duyệt tiền thứ tự** đúng của một **cây nhị phân tìm kiếm**.

**Ví dụ 1**

- Input: `preorder = [5,2,1,3,6]`
- Output: `true`
- Giải thích: BST gốc `5`, con trái `2` (có con `1`, `3`), con phải `6`; duyệt preorder ra `5, 2, 1, 3, 6`.

**Ví dụ 2**

- Input: `preorder = [5,2,6,1,3]`
- Output: `false`
- Giải thích: sau `5, 2` đến `6` (lớn hơn `5`) nghĩa là ta đã sang cây con phải của `5`; nhưng sau đó xuất hiện `1` (nhỏ hơn `5`) — vô lý.

**Ràng buộc:** `1 <= preorder.length <= 10^4`; `1 <= preorder[i] <= 10^4`; mọi phần tử của `preorder` **đôi một khác nhau**.

**Follow up:** bạn có thể làm chỉ với **bộ nhớ phụ hằng số** không?

In [1]:
from typing import List, Optional
from itertools import permutations
import random
import time

examples = [
    ([5, 2, 1, 3, 6], True),
    ([5, 2, 6, 1, 3], False),
]
for preorder, expected in examples:
    print(f"preorder={preorder} -> kỳ vọng {expected}")

preorder=[5, 2, 1, 3, 6] -> kỳ vọng True
preorder=[5, 2, 6, 1, 3] -> kỳ vọng False


## 3. Kiến thức nền

### 3.1. Cây nhị phân tìm kiếm (BST)

BST: với **mọi** nút `v`, **mọi** giá trị trong cây con trái của `v` đều `< v.val` và **mọi** giá trị trong cây con phải đều `> v.val` (các giá trị khác nhau). Chú ý "mọi" chứ không chỉ "con trực tiếp": đó là chỗ người mới hay nhầm.
#### Code demo

In [2]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right


def bst_insert(root, x):
    """Chèn x vào BST (không đệ quy), trả về gốc."""
    if root is None:
        return TreeNode(x)
    node = root
    while True:
        if x < node.val:
            if node.left is None:
                node.left = TreeNode(x)
                break
            node = node.left
        else:
            if node.right is None:
                node.right = TreeNode(x)
                break
            node = node.right
    return root


def preorder_of(root):
    out, stack = [], [root] if root else []
    while stack:
        node = stack.pop()
        out.append(node.val)
        if node.right:
            stack.append(node.right)
        if node.left:
            stack.append(node.left)
    return out


root = None
for x in [5, 2, 6, 1, 3]:
    root = bst_insert(root, x)
print("chèn 5,2,6,1,3 lần lượt -> preorder:", preorder_of(root))

chèn 5,2,6,1,3 lần lượt -> preorder: [5, 2, 1, 3, 6]


### 3.2. Duyệt tiền thứ tự (preorder)

Preorder: **thăm gốc trước**, rồi duyệt đệ quy cây con trái, rồi cây con phải. Với BST gốc `5`, con trái `2 (1, 3)`, con phải `6`: `5, 2, 1, 3, 6`. Dãy preorder luôn có dạng: `[gốc] + [mọi nút cây con trái] + [mọi nút cây con phải]`. Với BST, các nút cây con trái đều `< gốc`, các nút cây con phải đều `> gốc` ⇒ phần "nhỏ hơn gốc" **đứng liền nhau ngay sau gốc**, rồi mới đến phần "lớn hơn gốc".
#### Code demo

In [3]:
def preorder_rec(node):
    if node is None:
        return []
    return [node.val] + preorder_rec(node.left) + preorder_rec(node.right)


tree = TreeNode(5, TreeNode(2, TreeNode(1), TreeNode(3)), TreeNode(6))
seq = preorder_rec(tree)
print(seq)                                                   # [5, 2, 1, 3, 6]
root_val = seq[0]
print([x for x in seq[1:] if x < root_val], [x for x in seq[1:] if x > root_val])   # [2, 1, 3] [6]

[5, 2, 1, 3, 6]
[2, 1, 3] [6]


### 3.3. Stack (ngăn xếp) và stack đơn điệu

**Stack** = vào sau ra trước (`append`, `pop`, `stack[-1]`). **Stack đơn điệu giảm**: ta giữ các phần tử theo thứ tự **giảm dần từ đáy lên đỉnh**; khi gặp phần tử mới **lớn hơn** đỉnh, ta `pop` các đỉnh nhỏ hơn nó. Mỗi phần tử vào/ra tối đa một lần ⇒ tổng `O(n)`.
#### Code demo

In [4]:
stack = []
for x in [5, 2, 1, 3, 6]:
    popped = []
    while stack and stack[-1] < x:
        popped.append(stack.pop())
    stack.append(x)
    print(f"x={x}: đã pop {popped}, stack={stack}")

x=5: đã pop [], stack=[5]
x=2: đã pop [], stack=[5, 2]
x=1: đã pop [], stack=[5, 2, 1]
x=3: đã pop [1, 2], stack=[5, 3]
x=6: đã pop [3, 5], stack=[6]


### 3.4. Vô cực âm: `float("-inf")`

`float("-inf")` nhỏ hơn mọi số; dùng làm **giá trị khởi tạo** của cận dưới ("chưa có cận"). Mọi số nguyên đều `> -inf`. Tương tự `float("inf")` cho cận trên.
#### Code demo

In [5]:
low = float("-inf")
print(5 > low, -10 ** 9 > low)          # True True
print(min(3, float("inf")))             # 3

True True
3


### 3.5. Ý nghĩa của "cận dưới" `low` khi rẽ phải

Trong preorder, khi ta đang ở một nút `u` rồi đến phần tử **lớn hơn** `u`, nghĩa là ta đã **rời cây con trái của `u`** và đi vào **cây con phải** của một tổ tiên `a` (tổ tiên gần nhất có giá trị `< x`). Khi đó mọi phần tử sau đều phải thuộc cây con phải của `a`, tức **lớn hơn `a`**. Giá trị `a` chính là phần tử **cuối cùng bị `pop`** — nó trở thành **cận dưới `low`**: mọi phần tử về sau phải `> low`.
#### Code demo

In [6]:
seq = [5, 2, 6, 1, 3]
stack, low = [], float("-inf")
for x in seq:
    print(f"x={x}: low={low}", end=" ")
    if x < low:
        print("-> x < low: VI PHẠM (đã sang nhánh phải của", low, "mà vẫn có số nhỏ hơn)")
        break
    while stack and stack[-1] < x:
        low = stack.pop()
    stack.append(x)
    print(f"-> stack={stack}, low={low}")

x=5: low=-inf -> stack=[5], low=-inf
x=2: low=-inf -> stack=[5, 2], low=-inf
x=6: low=-inf -> stack=[6], low=5
x=1: low=5 -> x < low: VI PHẠM (đã sang nhánh phải của 5 mà vẫn có số nhỏ hơn)


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Ví dụ 2 `[5,2,6,1,3]`:

- `5` là gốc.
- `2 < 5`: nằm trong cây con trái của `5`.
- `6 > 5`: sang cây con phải của `5`. Từ giờ **mọi phần tử phải `> 5`**.
- `1`: nhỏ hơn `5` — **vi phạm** ⇒ `false`.

Ví dụ 1 `[5,2,1,3,6]`:

- `5` gốc; `2 < 5` rẽ trái; `1 < 2` rẽ trái nữa; `3`: lớn hơn `1` nên rời cây con trái của `1` và đi vào cây con phải của `2` (tổ tiên gần nhất nhỏ hơn `3`): từ giờ mọi phần tử `> 2`; `6`: lớn hơn `3`, `2`, `5` ⇒ cây con phải của `5`; từ giờ mọi phần tử `> 5`. Không có vi phạm ⇒ `true`.

**Cách nghĩ:** đi dọc dãy, nhớ **một cận dưới `low`** (mọi phần tử sau phải lớn hơn nó) và một **đường đi từ gốc** (các tổ tiên mà ta còn "đang ở bên trái" của chúng) — đó là **stack**.

## 5. Cách 1 — Brute Force / baseline: tách dãy thành "nhỏ hơn gốc" và "lớn hơn gốc" rồi đệ quy

**Ý tưởng.** Phần tử đầu là gốc `r`. Phần còn lại phải có dạng: **toàn bộ phần nhỏ hơn `r`** (cây con trái), rồi **toàn bộ phần lớn hơn `r`** (cây con phải). Tìm chỉ số đầu tiên `k` mà `seq[k] > r`; kiểm tra **mọi** phần tử từ `k` về sau đều `> r` (nếu có phần tử `< r` ở sau `k` thì sai); rồi đệ quy hợp lệ cho `seq[1:k]` và `seq[k:]`.
#### Code demo

In [7]:
def verify_split(seq):
    if len(seq) <= 1:
        return True
    root = seq[0]
    k = 1
    while k < len(seq) and seq[k] < root:                 # nửa đầu: cây con trái (nhỏ hơn gốc)
        k += 1
    for x in seq[k:]:                                      # nửa sau phải toàn lớn hơn gốc
        if x < root:
            return False
    return verify_split(seq[1:k]) and verify_split(seq[k:])


for preorder, expected in examples:
    print(preorder, "->", verify_split(preorder), "(kỳ vọng", expected, ")")

[5, 2, 1, 3, 6] -> True (kỳ vọng True )
[5, 2, 6, 1, 3] -> False (kỳ vọng False )


**Độ phức tạp (Cách 1).** Gọi `n = len(preorder)`. Mỗi cấp quét toàn dãy hiện tại; với dãy tăng dần `[1,2,...,n]` (cây lệch phải) độ sâu `n` ⇒ **Time `O(n²)`** (worst case), trung bình `O(n log n)` cho cây cân bằng; **Space `O(n)`** do lát cắt và độ sâu đệ quy (với `n = 10^4` còn dễ **tràn đệ quy** ở cây lệch).

**Nút thắt.** Ta **quét lại** các phần tử nhiều lần vì không nhớ "các cận" mà mỗi phần tử phải tuân theo. Chỉ cần truyền cận xuống.

## 6. Ý tưởng thuật toán / bước đột phá

**Truyền cận `(lo, hi)`:** một phần tử `x` ở một vị trí nào đó của cây phải nằm trong khoảng mở `(lo, hi)` do các tổ tiên quy định. Khi dựng BST theo preorder: nếu phần tử kế tiếp `v` thuộc `(lo, hi)` thì nó là **gốc của cây con này**; con trái nhận cận `(lo, v)`, con phải nhận `(v, hi)`. Dãy hợp lệ nếu **tiêu thụ hết** mọi phần tử.

**Stack đơn điệu:** thay vì đệ quy, giữ **stack các tổ tiên chưa rẽ phải** và **cận dưới `low`**. Phần tử mới `x`:

1. nếu `x < low` ⇒ `False`;
2. `pop` mọi đỉnh `< x` (ta đang rẽ phải tại tổ tiên cuối cùng bị pop) và cập nhật `low` thành giá trị `pop` cuối cùng;
3. `push x`.

Cuối cùng `True`. Mỗi phần tử vào ra stack một lần ⇒ `O(n)`.

## 7. Các cách tiếp theo

### Cách 2 — Đệ quy với cận `(lo, hi)` và chỉ số dùng chung

**Giải thích.** Con trỏ `idx` chỉ vào phần tử chưa dùng. `build(lo, hi)`: nếu `idx` còn trong dãy và `lo < pre[idx] < hi` thì `v = pre[idx]`, `idx += 1`, rồi `build(lo, v)` (cây con trái) và `build(v, hi)` (cây con phải). Sau khi dựng xong cây gốc `(-inf, +inf)`, hợp lệ nếu `idx == n`.
#### Code demo

In [8]:
def verify_bounds(preorder):
    n = len(preorder)
    idx = 0

    def build(lo, hi):
        nonlocal idx
        if idx < n and lo < preorder[idx] < hi:
            v = preorder[idx]
            idx += 1
            build(lo, v)               # cây con trái: giá trị trong (lo, v)
            build(v, hi)               # cây con phải: giá trị trong (v, hi)

    build(float("-inf"), float("inf"))
    return idx == n


for preorder, expected in examples:
    print(preorder, "->", verify_bounds(preorder), "(kỳ vọng", expected, ")")

[5, 2, 1, 3, 6] -> True (kỳ vọng True )
[5, 2, 6, 1, 3] -> False (kỳ vọng False )


**Độ phức tạp Cách 2:** mỗi phần tử được dùng đúng một lần, mỗi lời gọi `O(1)` ngoài lời gọi con: **Time `O(n)`**; **Space `O(h)`** ngăn xếp đệ quy với `h` là chiều cao (tối đa `n = 10^4` ở cây lệch ⇒ cần nâng giới hạn đệ quy hoặc dùng bản lặp).

### Cách 3 — Stack đơn điệu + cận dưới `low` (cách nộp chuẩn)

**Giải thích.** Như mục 6.
#### Code demo

In [9]:
def verify_stack(preorder):
    stack = []
    low = float("-inf")
    for x in preorder:
        if x < low:                                  # đã sang nhánh phải của `low` mà gặp số nhỏ hơn
            return False
        while stack and stack[-1] < x:               # x lớn hơn đỉnh: ta rẽ phải tại tổ tiên cuối cùng bị pop
            low = stack.pop()
        stack.append(x)
    return True


for preorder, expected in examples:
    print(preorder, "->", verify_stack(preorder), "(kỳ vọng", expected, ")")

[5, 2, 1, 3, 6] -> True (kỳ vọng True )
[5, 2, 6, 1, 3] -> False (kỳ vọng False )


In [10]:
# Code nộp LeetCode cho Cách 3
class Solution:
    def verifyPreorder(self, preorder: List[int]) -> bool:
        stk = []
        last = float("-inf")
        for x in preorder:
            if x < last:
                return False
            while stk and stk[-1] < x:
                last = stk.pop()
            stk.append(x)
        return True


print(Solution().verifyPreorder([5, 2, 1, 3, 6]), Solution().verifyPreorder([5, 2, 6, 1, 3]))

True False


**Độ phức tạp Cách 3:** **Time `O(n)`** (mỗi phần tử `push` một lần, `pop` tối đa một lần), **Space `O(n)`** cho stack (worst case dãy giảm dần).

### Cách 4 — Dùng chính mảng làm stack (follow-up: `O(1)` bộ nhớ phụ)

**Giải thích.** Stack chỉ chứa **một tiền tố của các phần tử đã duyệt** và luôn **không dài hơn** số phần tử đã đọc, nên có thể **ghi đè** lên chính mảng: con trỏ `top` là đỉnh; `pre[top]` là phần tử đỉnh. Để không làm hỏng đầu vào của người gọi ta chạy trên một bản sao khi test (trên LeetCode bạn có thể sửa trực tiếp).
#### Code demo

In [11]:
def verify_inplace(preorder):
    pre = list(preorder)                              # bản sao để không phá đầu vào; thuật toán chỉ dùng O(1) biến phụ
    top = -1                                          # chỉ số đỉnh stack nằm trong chính mảng pre
    low = float("-inf")
    for x in pre:                                     # đọc x trước khi ghi đè (vị trí ghi <= vị trí đọc)
        if x < low:
            return False
        while top >= 0 and pre[top] < x:
            low = pre[top]
            top -= 1
        top += 1
        pre[top] = x
    return True


for preorder, expected in examples:
    print(preorder, "->", verify_inplace(preorder), "(kỳ vọng", expected, ")")

[5, 2, 1, 3, 6] -> True (kỳ vọng True )
[5, 2, 6, 1, 3] -> False (kỳ vọng False )


**Độ phức tạp Cách 4:** **Time `O(n)`**, **Space `O(1)`** bộ nhớ phụ (ngoài mảng đầu vào được ghi đè; bản sao ở đây chỉ phục vụ kiểm thử, khi nộp bỏ dòng `list(...)` và dùng trực tiếp `preorder`).

> Cách 3 là bản nộp phổ biến; Cách 4 trả lời follow-up. Cách 2 giúp hiểu **vì sao** stack đúng (cận `(lo, hi)`).

## 8. TRACE từng bước

Ta trace Cách 3 trên `[5,2,1,3,6]` (hợp lệ) và `[5,2,6,1,3]` (không hợp lệ).
#### Code demo

In [12]:
def trace_stack(preorder):
    print("preorder =", preorder)
    stack, low = [], float("-inf")
    for x in preorder:
        if x < low:
            print(f"  x={x}: x < low={low} -> VI PHẠM, trả False")
            return False
        popped = []
        while stack and stack[-1] < x:
            low = stack.pop()
            popped.append(low)
        stack.append(x)
        print(f"  x={x}: pop {popped}, low={low}, stack={stack}")
    print("  hết dãy -> True")
    return True


trace_stack([5, 2, 1, 3, 6])
print()
trace_stack([5, 2, 6, 1, 3])

preorder = [5, 2, 1, 3, 6]
  x=5: pop [], low=-inf, stack=[5]
  x=2: pop [], low=-inf, stack=[5, 2]
  x=1: pop [], low=-inf, stack=[5, 2, 1]
  x=3: pop [1, 2], low=2, stack=[5, 3]
  x=6: pop [3, 5], low=5, stack=[6]
  hết dãy -> True

preorder = [5, 2, 6, 1, 3]
  x=5: pop [], low=-inf, stack=[5]
  x=2: pop [], low=-inf, stack=[5, 2]
  x=6: pop [2, 5], low=5, stack=[6]
  x=1: x < low=5 -> VI PHẠM, trả False


False

Bảng tóm tắt (`[5,2,1,3,6]`):

| `x` | `x < low`? | các số bị `pop` | `low` sau | stack sau |
|---|---|---|---|---|
| 5 | không | — | `-inf` | `[5]` |
| 2 | không | — | `-inf` | `[5, 2]` |
| 1 | không | — | `-inf` | `[5, 2, 1]` |
| 3 | không | `1`, `2` | `2` | `[5, 3]` |
| 6 | không | `3`, `5` | `5` | `[6]` |

Và `[5,2,6,1,3]`: tại `6` pop `2` rồi `5` ⇒ `low = 5`, stack `[6]`; tại `1`: `1 < 5` ⇒ **False**.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 3 (stack đơn điệu + cận dưới). Đúng `class Solution`, đúng `verifyPreorder(self, preorder: List[int]) -> bool`.

In [13]:
class Solution:
    def verifyPreorder(self, preorder: List[int]) -> bool:
        stk = []                                   # các tổ tiên chưa "rẽ phải", giá trị giảm dần từ đáy lên đỉnh
        last = float("-inf")                       # cận dưới: mọi phần tử sau phải > last
        for x in preorder:
            if x < last:                           # đã sang nhánh phải của `last` mà lại gặp số nhỏ hơn
                return False
            while stk and stk[-1] < x:             # x lớn hơn đỉnh: rẽ phải tại tổ tiên cuối cùng bị pop
                last = stk.pop()
            stk.append(x)
        return True


sol = Solution()
for preorder, expected in examples:
    assert sol.verifyPreorder(preorder) == expected
print("OK: 2 ví dụ của đề")

OK: 2 ví dụ của đề


## 10. Vì sao thuật toán đúng

- **Mô hình:** stack luôn chứa **đường đi từ gốc** tới phần tử vừa đọc, **chỉ gồm các tổ tiên mà ta đang ở nhánh trái của chúng** (giá trị giảm dần từ đáy lên đỉnh). Cận `low` là tổ tiên **sâu nhất** mà ta đã **rẽ phải** quanh nó.
- **Vì sao pop và cập nhật `low`:** phần tử `x` đến sau đỉnh `t`. Nếu `x < t`, `x` là con trái (trực tiếp hoặc gián tiếp) của `t`: cứ push. Nếu `x > t`, `x` **không** nằm trong cây con trái của `t` nên `t` "đóng" (pop); tương tự cho các tổ tiên nhỏ hơn `x`. Phần tử bị pop **cuối cùng** là tổ tiên sâu nhất nhỏ hơn `x`, và `x` là một nút trong **cây con phải** của nó ⇒ mọi phần tử sau phải `> ` tổ tiên đó.
- **Điều kiện `x < low` ⇒ sai:** nếu `x < low` thì `x` phải thuộc cây con trái của `low` (hoặc của một tổ tiên của `low`) nhưng preorder đã **rời cây con trái** của `low` rồi (đã duyệt phần trái và sang phải) — không thể quay lại.
- **Đủ điều kiện:** nếu không bao giờ vi phạm thì dãy khớp việc chèn lần lượt các phần tử vào BST (mỗi phần tử nằm đúng khoảng `(low, đỉnh)` cho phép), tức là preorder của chính BST thu được.
- **Cách 2 tương đương:** cận `(lo, hi)` của mỗi phần tử chính là `(low, đỉnh còn lại trên stack)`.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(preorder)`.

| Cách | Time | Space |
|---|---|---|
| 1. Tách "nhỏ/lớn hơn gốc" | `O(n²)` worst, `O(n log n)` cây cân bằng | `O(n)` (lát cắt + đệ quy) |
| 2. Đệ quy cận `(lo, hi)` | `O(n)` | `O(h)`, `h ≤ n` |
| 3. Stack đơn điệu + `low` | `O(n)` | `O(n)` |
| 4. Dùng mảng làm stack | `O(n)` | `O(1)` phụ |

Với `n = 10^4`: Cách 3/4 chỉ cỡ `2·10^4` thao tác. Cách 1 tệ nhất `~5·10^7` phép + nguy cơ đệ quy sâu. Không có khác biệt average/worst ở Cách 2–4 (mỗi phần tử vào/ra tối đa một lần).

## 12. Bộ test

`oracle` là brute force độc lập: **chèn lần lượt** các phần tử vào một BST rỗng (phép chèn chuẩn), rồi lấy **preorder** của cây thu được và **so với dãy đầu vào**. Dãy hợp lệ ⟺ chúng bằng nhau (nếu dãy là preorder của một BST `T`, chèn theo thứ tự đó dựng lại đúng `T`; ngược lại preorder của cây thu được khác dãy).
#### Code demo

In [14]:
def oracle(preorder):
    root = None
    for x in preorder:
        root = bst_insert(root, x)
    return preorder_of(root) == list(preorder)


impls = {
    "split": verify_split,
    "bounds": verify_bounds,
    "stack": verify_stack,
    "inplace": verify_inplace,
    "submit": sol.verifyPreorder,
}

for preorder, expected in examples:
    assert oracle(preorder) == expected
    for key, fn in impls.items():
        assert fn(list(preorder)) == expected, (key, preorder, fn(list(preorder)), expected)
print("PASS ví dụ của đề (mọi cách và oracle)")

PASS ví dụ của đề (mọi cách và oracle)


### 12.1. Các trường hợp biên
#### Code demo

In [15]:
edge_cases = [
    ([1], True, "một phần tử"),
    ([2, 1], True, "con trái"),
    ([1, 2], True, "con phải"),
    ([3, 2, 1], True, "dãy giảm dần: cây lệch trái"),
    ([1, 2, 3], True, "dãy tăng dần: cây lệch phải"),
    ([2, 1, 3], True, "cây hoàn hảo 3 nút"),
    ([2, 3, 1], False, "1 xuất hiện sau khi đã sang nhánh phải"),
    ([3, 1, 2], True, "1 rồi 2 > 1: 2 là con phải của 1"),
    ([3, 2, 4, 1], False, "1 xuất hiện sau 4"),
    ([4, 2, 3, 1], False, "1 xuất hiện sau 3 (đã rẽ phải tại 2)"),
    ([10000, 1], True, "giá trị biên của đề"),
    ([5, 4, 3, 2, 1, 6, 7, 8], True, "chuỗi giảm rồi chuỗi tăng"),
    ([5, 7, 6, 4], False, "4 xuất hiện sau 7"),
]
for preorder, expected, label in edge_cases:
    assert oracle(preorder) == expected, (label, oracle(preorder), expected)
    for key, fn in impls.items():
        assert fn(list(preorder)) == expected, (key, label, preorder, fn(list(preorder)), expected)
    print("PASS", label)

PASS một phần tử
PASS con trái
PASS con phải
PASS dãy giảm dần: cây lệch trái
PASS dãy tăng dần: cây lệch phải
PASS cây hoàn hảo 3 nút
PASS 1 xuất hiện sau khi đã sang nhánh phải
PASS 1 rồi 2 > 1: 2 là con phải của 1
PASS 1 xuất hiện sau 4
PASS 1 xuất hiện sau 3 (đã rẽ phải tại 2)
PASS giá trị biên của đề
PASS chuỗi giảm rồi chuỗi tăng
PASS 4 xuất hiện sau 7


### 12.2. Tính chất và phản ví dụ

- **Vét cạn mọi hoán vị** của `1..7` (5040 dãy): mọi cách phải khớp oracle; số dãy hợp lệ của `1..n` là **số Catalan** `C_n` (vì mỗi BST `n` nút khác nhau cho đúng một dãy preorder): `C_7 = 429`.
- **Dãy tăng dần / giảm dần** luôn hợp lệ.
- Một dãy hợp lệ mà **đổi chỗ hai phần tử** thường trở thành không hợp lệ; ta so với oracle để chắc chắn.
- **Phản ví dụ** cho lỗi chỉ so hai phần tử liền kề: `[5,2,6,1,3]` bị nhận nhầm `True`.
#### Code demo

In [16]:
from math import comb

for n in range(1, 8):
    valid = sum(1 for perm in permutations(range(1, n + 1)) if sol.verifyPreorder(list(perm)))
    catalan = comb(2 * n, n) // (n + 1)
    assert valid == catalan, (n, valid, catalan)
    for perm in permutations(range(1, n + 1)):
        p = list(perm)
        truth = oracle(p)
        for key, fn in impls.items():
            assert fn(list(p)) == truth, (key, p)
    print(f"n={n}: {valid} dãy hợp lệ = Catalan({n}) = {catalan}")
print("PASS vét cạn mọi hoán vị của 1..7: mọi cách khớp oracle; số dãy hợp lệ là số Catalan")


def sai_chi_so_lien_ke(preorder):
    for i in range(len(preorder) - 1):
        if preorder[i] > preorder[i + 1]:
            continue                                 # đi xuống trái: ổn
        # preorder[i + 1] > preorder[i]: coi như rẽ phải, ổn (SAI: không kiểm tra các số sau)
    return True


bad = [5, 2, 6, 1, 3]
print("phản ví dụ:", bad, "-> đúng", sol.verifyPreorder(bad), "| chỉ xét hai số liền kề:", sai_chi_so_lien_ke(bad))
assert sai_chi_so_lien_ke(bad) is True and sol.verifyPreorder(bad) is False

n=1: 1 dãy hợp lệ = Catalan(1) = 1
n=2: 2 dãy hợp lệ = Catalan(2) = 2
n=3: 5 dãy hợp lệ = Catalan(3) = 5
n=4: 14 dãy hợp lệ = Catalan(4) = 14
n=5: 42 dãy hợp lệ = Catalan(5) = 42
n=6: 132 dãy hợp lệ = Catalan(6) = 132
n=7: 429 dãy hợp lệ = Catalan(7) = 429
PASS vét cạn mọi hoán vị của 1..7: mọi cách khớp oracle; số dãy hợp lệ là số Catalan
phản ví dụ: [5, 2, 6, 1, 3] -> đúng False | chỉ xét hai số liền kề: True


### 12.3. Đối chiếu ngẫu nhiên (có seed) và dữ liệu lớn
#### Code demo

In [17]:
def random_valid_preorder(n, rng):
    values = rng.sample(range(1, 10001), n)
    root = None
    for x in values:
        root = bst_insert(root, x)
    return preorder_of(root)


random.seed(255)
rng = random.Random(2550)
for trial in range(1500):
    n = rng.randint(1, 40)
    seq = random_valid_preorder(n, rng)
    assert oracle(seq) is True
    for key, fn in impls.items():
        assert fn(list(seq)) is True, (key, seq)
    # đảo hai phần tử ngẫu nhiên và đối chiếu
    if n >= 2:
        i, j = rng.sample(range(n), 2)
        mutated = seq[:]
        mutated[i], mutated[j] = mutated[j], mutated[i]
        truth = oracle(mutated)
        for key, fn in impls.items():
            assert fn(list(mutated)) == truth, (key, mutated, fn(list(mutated)), truth)
print("PASS 1500 dãy hợp lệ ngẫu nhiên và các biến thể đổi chỗ hai phần tử: mọi cách khớp oracle")

# Dãy hoán vị hoàn toàn ngẫu nhiên (đa số không hợp lệ)
for trial in range(1500):
    n = rng.randint(1, 12)
    perm = rng.sample(range(1, 30), n)
    truth = oracle(perm)
    for key, fn in impls.items():
        assert fn(list(perm)) == truth, (key, perm)
print("PASS 1500 dãy ngẫu nhiên bất kỳ")

# n = 10^4: cây lệch (đệ quy sâu) và cây ngẫu nhiên; bản đệ quy cần nâng giới hạn đệ quy
import sys
sys.setrecursionlimit(50000)
increasing = list(range(1, 10001))
decreasing = list(range(10000, 0, -1))
big_random = random_valid_preorder(10000, rng)
big_bad = big_random[:]
big_bad[5000], big_bad[5001] = big_bad[5001], big_bad[5000]
for name, seq in [("tăng dần", increasing), ("giảm dần", decreasing), ("BST ngẫu nhiên", big_random), ("đã đổi chỗ", big_bad)]:
    results = {}
    for key in ("bounds", "stack", "inplace", "submit"):
        t0 = time.perf_counter()
        results[key] = impls[key](list(seq))
        dt = (time.perf_counter() - t0) * 1000
    assert len(set(results.values())) == 1, (name, results)
    print(f"{name:<15} kết quả {results['stack']}  (các cách khớp nhau)")
assert oracle(big_bad) == impls["stack"](list(big_bad))
print("PASS n = 10^4")

PASS 1500 dãy hợp lệ ngẫu nhiên và các biến thể đổi chỗ hai phần tử: mọi cách khớp oracle


PASS 1500 dãy ngẫu nhiên bất kỳ


tăng dần        kết quả True  (các cách khớp nhau)
giảm dần        kết quả True  (các cách khớp nhau)


BST ngẫu nhiên  kết quả True  (các cách khớp nhau)
đã đổi chỗ      kết quả False  (các cách khớp nhau)


PASS n = 10^4


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Chỉ so sánh hai phần tử liền kề
Xem `sai_chi_so_lien_ke` ở phần test: bản này luôn trả `True`. **Vì sao sai:** ràng buộc BST là **toàn cục** (mọi nút cây con phải `>` tổ tiên), không chỉ giữa hai nút liền kề. **Code đúng:** nhớ cận dưới `low`.

### Lỗi 2: Quên cập nhật `low` khi `pop`
#### Code sai

In [18]:
def sai_khong_cap_nhat_low(preorder):
    stack = []
    low = float("-inf")
    for x in preorder:
        if x < low:
            return False
        while stack and stack[-1] < x:
            stack.pop()                              # SAI: không gán low = giá trị vừa pop
        stack.append(x)
    return True


for seq in ([5, 2, 6, 1, 3], [5, 2, 1, 3, 6]):
    print(seq, "đúng:", sol.verifyPreorder(seq), "| quên cập nhật low:", sai_khong_cap_nhat_low(seq))

[5, 2, 6, 1, 3] đúng: False | quên cập nhật low: True
[5, 2, 1, 3, 6] đúng: True | quên cập nhật low: True


**Vì sao sai:** `low` luôn là `-inf` nên điều kiện `x < low` không bao giờ đúng. **Code đúng:** `low = stack.pop()`.

### Lỗi 3: Gán `low` bằng đỉnh mới thay vì giá trị vừa `pop`
#### Code sai

In [19]:
def sai_low_sai_gia_tri(preorder):
    stack = []
    low = float("-inf")
    for x in preorder:
        if x < low:
            return False
        while stack and stack[-1] < x:
            stack.pop()
            low = x                                  # SAI: low phải là phần tử bị pop (tổ tiên), không phải x
        stack.append(x)
    return True


for seq in ([5, 2, 1, 3, 6], [10, 5, 1, 8, 6]):
    print(seq, "đúng:", sol.verifyPreorder(seq), "| sai:", sai_low_sai_gia_tri(seq))

[5, 2, 1, 3, 6] đúng: True | sai: True
[10, 5, 1, 8, 6] đúng: True | sai: False


**Vì sao sai:** với `[10,5,1,8,6]` (hợp lệ: `5` con trái của `10`, `1` con trái của `5`, `8` con phải của `5`, `6` con trái của `8`), tại `8` tổ tiên cuối cùng bị pop là `5`, nên cận dưới đúng là `5`; bản sai đặt `low = 8` làm `6` bị loại nhầm. Quy tắc đúng: `low` là **tổ tiên** mà ta vừa rời bỏ cây con trái, không phải phần tử hiện tại.

### Lỗi 4: Dùng `<=` / thứ tự so sánh sai (và nhầm với inorder)
#### Code sai

In [20]:
def sai_kiem_tra_inorder(seq):
    return all(seq[i] < seq[i + 1] for i in range(len(seq) - 1))      # SAI: đó là điều kiện của dãy INORDER của BST


for seq in ([5, 2, 1, 3, 6], [1, 2, 3, 5, 6]):
    print(seq, "preorder hợp lệ:", sol.verifyPreorder(seq), "| coi như inorder:", sai_kiem_tra_inorder(seq))

[5, 2, 1, 3, 6] preorder hợp lệ: True | coi như inorder: False
[1, 2, 3, 5, 6] preorder hợp lệ: True | coi như inorder: True


**Vì sao sai:** dãy duyệt **inorder** của BST luôn **tăng dần**; preorder thì không (gốc đứng đầu rồi mới đến con trái nhỏ hơn). Đề hỏi **preorder**.

### Lỗi 5: Tách dãy sai chỗ ở cách đệ quy (chỉ kiểm tra phần tử đầu của nửa sau)
#### Code sai

In [21]:
def sai_tach_day(seq):
    if len(seq) <= 1:
        return True
    root = seq[0]
    k = 1
    while k < len(seq) and seq[k] < root:
        k += 1
    # SAI: không kiểm tra mọi phần tử của nửa sau đều > root
    return sai_tach_day(seq[1:k]) and sai_tach_day(seq[k:])


bad = [5, 2, 6, 1, 3]
print("đúng:", sol.verifyPreorder(bad), "| tách dãy sai:", sai_tach_day(bad))

đúng: False | tách dãy sai: True


**Vì sao sai:** sau khi tách `[2]` (trái) và `[6, 1, 3]` (phải), nửa phải có `1 < 5` nhưng thuật toán không kiểm tra điều đó. Phải **xác nhận mọi phần tử từ chỗ tách về sau đều `> gốc`** (như `verify_split`).

## 14. Mẹo làm nhanh & nhận dạng pattern

- Thấy "**kiểm tra dãy preorder/postorder có hợp lệ cho BST**" → **stack đơn điệu + cận dưới** (preorder) hoặc cận trên (postorder đảo).
- Quy tắc "**rẽ phải ⇒ cận dưới tăng**": khi gặp số **lớn hơn** đỉnh, ta `pop` hết các số nhỏ hơn; số cuối cùng bị pop trở thành cận dưới.
- Mọi bài "**ràng buộc toàn cục kiểu BST**" đều nghĩ đến **truyền cận `(lo, hi)` xuống** (như *Validate Binary Search Tree* 98).
- Follow-up **`O(1)` bộ nhớ phụ:** stack nằm trọn trong tiền tố của chính mảng (con trỏ ghi ≤ con trỏ đọc).
- Số dãy preorder hợp lệ của một tập `n` giá trị khác nhau là **số Catalan** — dùng để tự kiểm tra bằng vét cạn hoán vị.
- Test nhanh: dãy tăng dần, giảm dần, `[5,2,6,1,3]` (số nhỏ xuất hiện sau khi đã rẽ phải).

## 15. Biến thể của bài

1. **Validate Binary Search Tree** (LeetCode 98): kiểm tra một cây có phải BST.
2. **Construct BST from Preorder Traversal** (1008): dựng cây từ preorder hợp lệ (cận `(lo, hi)` hoặc stack).
3. **Verify Postorder** (kiểm tra dãy postorder của BST): đảo dãy rồi đối xứng với preorder.
4. **Serialize and Deserialize BST** (449): lưu cây bằng preorder.
5. **Count BSTs** (96): số BST `n` nút khác nhau — số Catalan.
#### Code demo

In [22]:
def verify_postorder(postorder):
    """Postorder của BST: đảo ngược được 'gốc, phải, trái' -> dùng stack với cận trên."""
    stack, high = [], float("inf")
    for x in reversed(postorder):
        if x > high:
            return False
        while stack and stack[-1] > x:
            high = stack.pop()
        stack.append(x)
    return True


def build_bst_from_preorder(preorder):
    idx = 0

    def build(lo, hi):
        nonlocal idx
        if idx < len(preorder) and lo < preorder[idx] < hi:
            node = TreeNode(preorder[idx])
            idx += 1
            node.left = build(lo, node.val)
            node.right = build(node.val, hi)
            return node
        return None

    return build(float("-inf"), float("inf"))


print(verify_postorder([1, 3, 2, 6, 5]), verify_postorder([1, 6, 3, 2, 5]))                 # True False
print(preorder_of(build_bst_from_preorder([8, 5, 1, 7, 10, 12])))                           # [8, 5, 1, 7, 10, 12]

True False
[8, 5, 1, 7, 10, 12]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Tách nhỏ/lớn hơn gốc | đệ quy trên lát cắt | `O(n²)` worst | `O(n)` | hiểu cấu trúc preorder, oracle nhỏ |
| 2. Đệ quy cận `(lo, hi)` | dựng ngầm BST, kiểm tra tiêu thụ hết dãy | `O(n)` | `O(h)` | giải thích rõ nhất tính BST |
| 3. Stack đơn điệu + `low` | stack các tổ tiên, `low` = tổ tiên đã rẽ phải | `O(n)` | `O(n)` | **cách nộp chuẩn** |
| 4. Mảng làm stack | tái dùng tiền tố mảng | `O(n)` | `O(1)` | trả lời follow-up |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `Solution().verifyPreorder([2, 1, 3])`; (b) `Solution().verifyPreorder([2, 3, 1])`; (c) `Solution().verifyPreorder([1, 2, 3, 4])`; (d) `preorder_of` của cây dựng bằng chèn `3, 1, 2`.

**Bài 2 (tính bằng tay).** Trace Cách 3 trên `[8, 5, 1, 7, 10, 12]` và trên `[8, 5, 1, 7, 6, 12]`: ghi `x`, các số bị pop, `low`, `stack` sau mỗi bước.

**Bài 3 (điền TODO).** Hoàn thiện `verify_todo(preorder)` theo Cách 3.
#### Code demo

In [23]:
def verify_todo(preorder):
    stack = []
    low = float("-inf")
    # TODO 1: duyệt x trong preorder
    # TODO 2: nếu x < low -> trả False
    # TODO 3: while stack và stack[-1] < x: low = stack.pop()
    # TODO 4: stack.append(x)
    # TODO 5: hết vòng lặp trả True
    pass


# Tự làm xong rồi mới mở test:
# for preorder, expected in examples:
#     assert verify_todo(preorder) == expected

**Bài 4 (tự viết lại).** Viết `verify_bounds2(preorder)` bằng đệ quy cận `(lo, hi)`, **không nhìn lại** Cách 2; so với oracle trên mọi hoán vị của `1..6`.

**Bài 5 (biến thể / mini challenge).** Viết `verify_postorder2(postorder)` (kiểm tra dãy **postorder** của BST) mà **không** đảo ngược dãy: duyệt từ **phải sang trái** với stack và cận trên `high`; so với oracle "chèn theo thứ tự đảo ngược rồi so postorder". Thử `[1, 3, 2, 6, 5]` → `True`, `[1, 6, 3, 2, 5]` → `False`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: sau khi sang nhánh phải (số lớn hơn gốc), không còn số nhỏ hơn gốc. Bài 2: `pop` mọi đỉnh nhỏ hơn `x`, ghi lại giá trị `pop` cuối làm `low`. Bài 5: preorder đảo ngược thành "gốc, phải, trái".
- **Mức 2 (vừa):** Bài 1: `True`, `False`, `True`, cây chèn `3, 1, 2` có preorder `[3, 1, 2]`. Bài 2: `[8,5,1,7,10,12]`: tại `7` pop `1`, `5` ⇒ `low = 5`, stack `[8,7]`; tại `10` pop `7`, `8` ⇒ `low = 8`; tại `12` pop `10` ⇒ `low = 10` ⇒ `True`. Với `[8,5,1,7,6,12]`: tại `6` có `6 > low = 5` và `6 < 7` nên push (`6` là con trái của `7`); các bước còn lại không vi phạm. Bài 3: khung Cách 3.
- **Mức 3 (sâu):** Bài 2 (đáp án): cả hai dãy đều hợp lệ (`True`); `[8,5,1,7,6,12]` hợp lệ vì `6` là con trái của `7`. Dãy sai chỉ khi sau khi rẽ phải lại có số nhỏ hơn cận, ví dụ `[8,5,1,7,4,12]` (vì `4 < low = 5`). Bài 4: `idx` dùng chung; kết quả `idx == n`. Bài 5: cận trên `high = +inf`; với `x` duyệt từ cuối: nếu `x > high` sai; `while stack and stack[-1] > x: high = stack.pop()`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [24]:
# Bài 1
assert Solution().verifyPreorder([2, 1, 3]) is True
assert Solution().verifyPreorder([2, 3, 1]) is False
assert Solution().verifyPreorder([1, 2, 3, 4]) is True
r = None
for x in [3, 1, 2]:
    r = bst_insert(r, x)
assert preorder_of(r) == [3, 1, 2]

# Bài 2
assert Solution().verifyPreorder([8, 5, 1, 7, 10, 12]) is True and oracle([8, 5, 1, 7, 10, 12]) is True
assert Solution().verifyPreorder([8, 5, 1, 7, 6, 12]) == oracle([8, 5, 1, 7, 6, 12])
print("[8,5,1,7,6,12] ->", Solution().verifyPreorder([8, 5, 1, 7, 6, 12]))


# Bài 3
def verify_todo_ans(preorder):
    stack = []
    low = float("-inf")
    for x in preorder:
        if x < low:
            return False
        while stack and stack[-1] < x:
            low = stack.pop()
        stack.append(x)
    return True


for preorder, expected in examples:
    assert verify_todo_ans(preorder) == expected


# Bài 4
def verify_bounds2(preorder):
    n, idx = len(preorder), 0

    def build(lo, hi):
        nonlocal idx
        if idx < n and lo < preorder[idx] < hi:
            v = preorder[idx]
            idx += 1
            build(lo, v)
            build(v, hi)

    build(float("-inf"), float("inf"))
    return idx == n


for perm in permutations(range(1, 7)):
    assert verify_bounds2(list(perm)) == oracle(list(perm))


# Bài 5
def verify_postorder2(postorder):
    stack, high = [], float("inf")
    for i in range(len(postorder) - 1, -1, -1):
        x = postorder[i]
        if x > high:
            return False
        while stack and stack[-1] > x:
            high = stack.pop()
        stack.append(x)
    return True


assert verify_postorder2([1, 3, 2, 6, 5]) is True and verify_postorder2([1, 6, 3, 2, 5]) is False
print("PASS đáp án bài tập")

[8,5,1,7,6,12] -> True
PASS đáp án bài tập


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (stack đơn điệu + cận dưới)

```python
class Solution:
    def verifyPreorder(self, preorder: List[int]) -> bool:
        stk, last = [], float("-inf")
        for x in preorder:
            if x < last:
                return False
            while stk and stk[-1] < x:
                last = stk.pop()
            stk.append(x)
        return True
```

**Checklist**

- [ ] Nhớ quy tắc: gặp số **lớn hơn** đỉnh ⇒ `pop` các số nhỏ hơn; **số bị pop cuối cùng = cận dưới `last`**?
- [ ] Kiểm tra `x < last` **trước** khi xử lý `x` (và trả `False` ngay)?
- [ ] `last` được gán bằng **giá trị vừa pop** (không phải `x`)?
- [ ] Hiểu ràng buộc BST là **toàn cục** (không chỉ hai phần tử liền kề)?
- [ ] Biết follow-up `O(1)`: dùng chính mảng làm stack?
- [ ] Đã thử `[5,2,6,1,3]` (False), `[5,2,1,3,6]` (True), dãy tăng dần, dãy giảm dần?
- [ ] Đã đối chiếu với brute force "chèn vào BST rồi so preorder" và vét cạn mọi hoán vị nhỏ?
- [ ] Skeleton đúng `class Solution` / `def verifyPreorder(self, preorder: List[int]) -> bool`?